# Laboratorium 6: Mechanizm uwagi i transformery w przetwarzaniu tekstu

Kontynuujemy laboratorium 5 na tych samych danych: wydźwięk recenzji z PolEmo 2.0-IN w czterech klasach (minus, plus, amb, zero). Najpierw policzycie mechanizm uwagi ręcznie w numpy, potem zobaczycie, jak tekst dzieli tokenizer HerBERT, a na końcu dostroicie (fine-tuning) cały model HerBERT do klasyfikacji i porównacie go z modelami z laboratorium 5.

Środowisko: Google Colab z GPU (Środowisko wykonawcze, Zmień typ środowiska wykonawczego, T4 GPU). Bez GPU fine-tuning HerBERT jest praktycznie niewykonalny w czasie zajęć.

Ważne: ten notebook działa na Keras 3 z backendem torch, a nie TensorFlow. Biblioteka transformers w wersji 5 nie obsługuje już TensorFlow, a HerBERT nie ma gotowego presetu w KerasHub. Backend wybiera się przed pierwszym `import keras` i nie da się go zmienić w trakcie sesji. Zacznijcie więc od świeżego środowiska wykonawczego. Jeśli w tej sesji Colaba uruchamialiście wcześniej inny notebook z Keras (np. z laboratorium 5), zrestartujcie sesję: Środowisko wykonawcze, Uruchom ponownie sesję (ang. Runtime, Restart session). Komórka importów sprawdza backend i zatrzyma notebook z komunikatem, jeśli nie jest to torch.

Dane: PolEmo 2.0 (Kocoń i in., 2019) w wersji z benchmarku KLEJ, pliki CSV z Hugging Face (`allegro/klej-polemo2-in`). Źródła podają różne licencje: repozytorium CLARIN-PL (korpus pierwotny) CC BY 4.0, strona KLEJ dla wersji w benchmarku CC BY-NC-SA 4.0, karta na Hugging Face ma tag CC BY-SA 4.0. Do użytku dydaktycznego wystarcza każda z nich; przy innym użyciu przyjmijcie najbardziej restrykcyjną (NC, bez użytku komercyjnego). Model: `allegro/herbert-base-cased` (Mroczkowski i in., 2021), licencja CC BY 4.0.

Komórka importów ustawia backend torch. Linijka `os.environ["KERAS_BACKEND"] = "torch"` musi się wykonać przed pierwszym `import keras` w sesji.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # musi być przed pierwszym import keras w tej sesji

import time

import keras
from keras import layers, ops

if keras.backend.backend() != "torch":
    raise RuntimeError(
        f"Keras działa na backendzie '{keras.backend.backend()}', a ten notebook wymaga backendu 'torch'. "
        "Keras został zaimportowany wcześniej w tej sesji. Uruchomcie sesję ponownie "
        "(Środowisko wykonawcze, Uruchom ponownie sesję) i wykonajcie notebook od pierwszej komórki.")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", 120)
T_START = time.time()

print("Keras", keras.__version__, "| backend", keras.backend.backend(),
      "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "brak, obliczenia na CPU")

Wszystkie parametry, od których zależy czas obliczeń, są w jednej komórce. W Colabie z GPU zostawcie `TRYB_SKROCONY = False`. Tryb skrócony służy tylko do sprawdzenia, czy notebook działa od początku do końca na komputerze bez GPU: skraca sekwencje i trenuje na małym podzbiorze, więc jego wyniki są znacznie gorsze od pełnego fine-tuningu.

In [ ]:
TRYB_SKROCONY = False  # True tylko do szybkiego sprawdzenia na CPU; w Colabie z GPU: False

MAX_LEN = 256    # długość sekwencji w tokenach HerBERT (zadanie 2)
BATCH = 16       # batch przy fine-tuningu HerBERT (przy braku pamięci GPU: 8)
EPOKI = 3        # górna granica liczby epok fine-tuningu, EarlyStopping może skończyć wcześniej
LR = 2e-5        # learning rate optymalizatora AdamW przy fine-tuningu
N_TRAIN = None   # liczba recenzji treningowych do fine-tuningu, None = wszystkie (5783)
N_VALID = None   # liczba recenzji walidacyjnych, None = wszystkie (723)
N_CECHY = None   # zadania rozszerzone, zadanie B: recenzje treningowe dla zamrożonego HerBERT, None = wszystkie
EPOKI_MINI = 10  # zadania rozszerzone, zadanie A: górna granica epok mini-transformera
BATCH_MINI = 64

if TRYB_SKROCONY:
    MAX_LEN, EPOKI, N_TRAIN, N_VALID, N_CECHY, EPOKI_MINI = 128, 2, 320, 200, 1000, 5
print(f"TRYB_SKROCONY={TRYB_SKROCONY}: MAX_LEN={MAX_LEN}, BATCH={BATCH}, EPOKI={EPOKI}, "
      f"N_TRAIN={N_TRAIN}, N_VALID={N_VALID}, N_CECHY={N_CECHY}, EPOKI_MINI={EPOKI_MINI}")

Gotowa komórka: wczytanie danych tak samo jak w laboratorium 5. Kolumna `y` zawiera numer klasy według listy `KLASY`. Funkcja `podzbior` losuje podzbiór z zachowaniem proporcji klas; przy `N_TRAIN = None` i `N_VALID = None` zwraca cały zbiór, więc `train_ft` i `valid_ft` (dane do fine-tuningu) są wtedy po prostu `train_df` i `valid_df`. Test jest zawsze pełny.

In [ ]:
URL_IN = "https://huggingface.co/datasets/allegro/klej-polemo2-in/resolve/main/{}.csv"
train_df = pd.read_csv(URL_IN.format("train"))
valid_df = pd.read_csv(URL_IN.format("valid"))
test_df = pd.read_csv(URL_IN.format("test"))

ETYKIETY = {"__label__meta_minus_m": "minus", "__label__meta_plus_m": "plus",
            "__label__meta_amb": "amb", "__label__meta_zero": "zero"}
KLASY = ["minus", "plus", "amb", "zero"]
OPIS_KLAS = {"minus": "negatywna", "plus": "pozytywna", "amb": "ambiwalentna", "zero": "neutralna"}

for df in (train_df, valid_df, test_df):
    df["klasa"] = df["target"].map(ETYKIETY)
    df["y"] = df["klasa"].map(KLASY.index)
    assert df["klasa"].notna().all()


def podzbior(df, n):
    if n is None or n >= len(df):
        return df
    return train_test_split(df, train_size=n, stratify=df["y"], random_state=RANDOM_STATE)[0].reset_index(drop=True)


train_ft, valid_ft = podzbior(train_df, N_TRAIN), podzbior(valid_df, N_VALID)
y_train_ft, y_valid_ft, y_test = (df["y"].to_numpy() for df in (train_ft, valid_ft, test_df))
print("train", len(train_df), "| valid", len(valid_df), "| test", len(test_df))
print("do fine-tuningu: train", len(train_ft), "| valid", len(valid_ft))
print("klasy w train_ft:", train_ft["klasa"].value_counts().reindex(KLASY).to_dict())

Gotowe funkcje pomocnicze, jak w laboratorium 5. `ocen` liczy accuracy i macro F1 i zapisuje wynik w słowniku `wyniki`. `przewidz` zwraca numery klas. `rysuj_uwage` rysuje macierz wag uwagi z podpisanymi tokenami.

In [ ]:
wyniki = {}


def ocen(nazwa, zbior, y_true, y_pred):
    wiersz = {"model": nazwa, "zbiór": zbior,
              "accuracy": accuracy_score(y_true, y_pred),
              "macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0)}
    wyniki[(nazwa, zbior)] = wiersz
    print(f"{nazwa} [{zbior}]: accuracy {wiersz['accuracy']:.4f}, macro F1 {wiersz['macro F1']:.4f}")
    return wiersz


def tabela_wynikow(zbior):
    df = pd.DataFrame([w for w in wyniki.values() if w["zbiór"] == zbior])
    return df.sort_values("macro F1", ascending=False).reset_index(drop=True).round(4)


def przewidz(model, X, batch_size=64):
    return model.predict(X, batch_size=batch_size, verbose=0).argmax(axis=1)


def rysuj_historie(historia, tytul):
    h = historia.history
    epoki = np.arange(1, len(h["loss"]) + 1)
    fig, osie = plt.subplots(1, 2, figsize=(11, 3.5))
    for os_, klucz, nazwa in [(osie[0], "loss", "strata"), (osie[1], "accuracy", "accuracy")]:
        os_.plot(epoki, h[klucz], marker="o", label="trening")
        os_.plot(epoki, h["val_" + klucz], marker="o", label="walidacja")
        os_.set_xlabel("epoka")
        os_.set_ylabel(nazwa)
        os_.set_xticks(epoki)
        os_.legend()
    fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()


def rysuj_macierz(y_true, y_pred, tytul, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=range(len(KLASY))),
                           display_labels=KLASY).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_xlabel("klasa przewidziana")
    ax.set_ylabel("klasa prawdziwa")
    ax.set_title(tytul)


def rysuj_uwage(wagi, tokeny, tytul, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4.2))
    obraz = ax.imshow(wagi, cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(tokeny)), tokeny, rotation=45, ha="right")
    ax.set_yticks(range(len(tokeny)), tokeny)
    ax.set_xlabel("klucz (na który token patrzymy)")
    ax.set_ylabel("zapytanie (który token patrzy)")
    ax.set_title(tytul)
    plt.colorbar(obraz, ax=ax, fraction=0.046)

## Zadanie 1: Uwaga w numpy

Scaled dot-product attention: $\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$, gdzie $Q = XW_Q$, $K = XW_K$, $V = XW_V$, a $X$ to wektory tokenów (jeden wiersz na token).

1. Gotowa komórka tworzy sześć „tokenów” z losowymi wektorami ($d_{model} = 8$), macierze $W_Q, W_K, W_V$ ($8 \times 4$), macierz wyjściową $W_O$ ($4 \times 8$) i maskę, w której ostatni token to padding.
2. Napiszcie `softmax(z)` (po ostatniej osi, z odjęciem maksimum dla stabilności) i `uwaga(Q, K, V, maska)`, która zwraca wyjście i macierz wag. Pozycjom paddingu w kluczach przed softmaxem wpiszcie bardzo małą liczbę (np. -1e9).
3. Sprawdźcie kształty, to, że każdy wiersz wag sumuje się do 1, i to, że kolumna paddingu ma wagi 0. Narysujcie macierz wag (`rysuj_uwage`).
4. Porównajcie wynik z `keras.layers.MultiHeadAttention(num_heads=1, key_dim=4, use_bias=False)`, do której wpiszecie te same macierze (`set_weights`). Wyjście warstwy to `uwaga(...) @ W_O`.
5. Sprawdźcie, co dzieje się z softmaxem bez dzielenia przez $\sqrt{d_k}$ przy $d_k$ równym 4, 64 i 512.

Po co dzielimy przez $\sqrt{d_k}$? Czym różni się macierz wag uwagi od wag warstwy Dense: co jest tu parametrem uczonym, a co zależy od wejścia?

In [ ]:
TOKENY = ["hotel", "był", "czysty", "ale", "drogi", "<pad>"]
D_MODEL, D_K = 8, 4
rng = np.random.default_rng(RANDOM_STATE)
X_toy = rng.normal(size=(len(TOKENY), D_MODEL)).astype("float32")  # wektory tokenów (jeden wiersz na token)
W_Q, W_K, W_V = (rng.normal(size=(D_MODEL, D_K)).astype("float32") for _ in range(3))
W_O = rng.normal(size=(D_K, D_MODEL)).astype("float32")
maska_toy = np.array([True, True, True, True, True, False])  # False = padding
print("X:", X_toy.shape, "| W_Q:", W_Q.shape, "| W_O:", W_O.shape)

In [ ]:
# TODO: softmax(z): odejmijcie maksimum po ostatniej osi, exp, podzielcie przez sumę
# TODO: uwaga(Q, K, V, maska=None): oceny = Q @ K.T / sqrt(d_k); padding w kluczach -> -1e9; wagi = softmax(oceny)
#       zwraca (wagi @ V, wagi)
# TODO: Q, K, V = X_toy @ W_Q, ...; Y_toy, A_toy = uwaga(Q, K, V, maska_toy)
# TODO: kształty, sumy wierszy, kolumna <pad>; rysuj_uwage(A_toy, TOKENY, "...")

In [ ]:
# TODO: mha = layers.MultiHeadAttention(num_heads=1, key_dim=D_K, use_bias=False)
# TODO: x = X_toy[None]; maska_mha o kształcie (1, 6, 6) z maska_toy (np.broadcast_to)
# TODO: pierwsze wywołanie mha(x, x, attention_mask=maska_mha) tworzy wagi; wypiszcie ich kształty
# TODO: mha.set_weights([W_Q[:, None, :], W_K[:, None, :], W_V[:, None, :], W_O[None]])
# TODO: wywołanie z return_attention_scores=True, porównanie z Y_toy @ W_O i A_toy (ops.convert_to_numpy)

In [ ]:
# TODO: dla d_k w (4, 64, 512): q = rng.normal(size=(1000, d)), k = rng.normal(size=(10, d)), oceny = q @ k.T
# TODO: odchylenie standardowe ocen, średnia największa waga softmax w wierszu bez skalowania i z / sqrt(d)

Odpowiedzi do zadania 1:

1. 
2. 
3. 
4. 

## Zadanie 2: Tokenizacja subword HerBERT

1. Wczytajcie tokenizer HerBERT (gotowa komórka). Zobaczcie jego tokeny specjalne i ich indeksy.
2. Stokenizujcie kilka zdań (`tokenizer.tokenize`): fleksja, brak polskich znaków, literówki, bardzo rzadkie słowo. Dla jednego zdania pokażcie pełne kodowanie z tokenami specjalnymi, paddingiem do 16 i `attention_mask`.
3. Policzcie odsetek tokenów `<unk>` i średnią liczbę tokenów na słowo na walidacji. Porównajcie z odsetkiem OOV ze słownika z laboratorium 5.
4. Narysujcie histogram długości recenzji treningowych w tokenach HerBERT, podajcie percentyle i odsetek recenzji dłuższych niż 128, 256 i 512 tokenów.
5. Napiszcie `koduj(teksty)`, która zwraca listę `[input_ids, attention_mask]` (tablice int32 o długości `MAX_LEN`, padding i obcięcie), i zakodujcie `train_ft`, `valid_ft` i `test_df`.

Dlaczego tokenizer subword prawie nie ma tokenów spoza słownika? Co oznacza `</w>` na końcu tokenu? Czy `MAX_LEN = 256` to rozsądny wybór i co tracimy przy obcięciu?

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
print(type(tokenizer).__name__, "| rozmiar słownika:", tokenizer.vocab_size)
print("tokeny specjalne:", {t: tokenizer.convert_tokens_to_ids(t) for t in tokenizer.all_special_tokens})

In [ ]:
ZDANIA = [
    "Lekarz był bardzo miły, polecam!",
    "lekarz lekarza lekarzowi lekarzem lekarzami",
    "Recepcjonistka byla niemila, a pokoj brudny",
    "Polecm tego lekrza, swietny fachofiec",
    "Konstantynopolitańczykowianeczka nie polecam",
]
# TODO: dla każdego zdania: tokenizer.tokenize(zdanie), liczba słów i tokenów
# TODO: przyklad = tokenizer(ZDANIA[0], padding="max_length", truncation=True, max_length=16)
#       wypiszcie input_ids, tokenizer.convert_ids_to_tokens(...) i attention_mask

In [ ]:
# TODO: kody_valid = tokenizer(list(valid_df["sentence"]))["input_ids"]  (ostrzeżenie o > 512 tokenach jest tu oczekiwane)
# TODO: liczba tokenów (bez <s> i </s>), liczba <unk> (tokenizer.unk_token_id), tokeny na słowo (słowa po spacjach)
# TODO: dl_herbert = długości w tokenach dla train_df, describe z percentylami 0.5, 0.9, 0.95, 0.99
# TODO: odsetek recenzji dłuższych niż 128, 256, 512; histogram z liniami pionowymi

In [ ]:
# TODO: koduj(teksty): tokenizer(list(teksty), padding="max_length", truncation=True, max_length=MAX_LEN,
#       return_tensors="np"); zwróćcie [input_ids, attention_mask] jako int32
# TODO: X_train, X_valid, X_test = koduj(...) dla train_ft, valid_ft, test_df; kształty
X_train = X_valid = X_test = None

Odpowiedzi do zadania 2:

1. 
2. 
3. 
4. 

## Zadanie 3: Fine-tuning HerBERT

Gotowa komórka definiuje moduł PyTorch `EnkoderHerBERT`: wczytuje `allegro/herbert-base-cased` przez `AutoModel` i zwraca wektor pierwszego tokenu (`<s>`, odpowiednik `[CLS]` z BERT) z ostatniej warstwy, 768 liczb na recenzję. Przy wczytaniu transformers wypisuje raport o wagach `cls.*` oznaczonych jako UNEXPECTED: to głowa do pretreningu (przewidywanie zamaskowanych słów), której nie używamy, więc tak ma być.

`keras.layers.TorchModuleWrapper` przekazuje modułowi jeden argument, dlatego dwa wejścia podajemy jako listę `[input_ids, attention_mask]`, a `forward` ją rozpakowuje. Pooler BERT wyłączamy (`add_pooling_layer=False`), bo z niego nie korzystamy; inaczej Keras ostrzega, że jego wagi nie mają gradientów.

1. Napiszcie `zbuduj_herbert()`: dwa wejścia `keras.Input(shape=(MAX_LEN,), dtype="int32")` o nazwach `input_ids` i `attention_mask`, `TorchModuleWrapper(EnkoderHerBERT(), output_shape=(None, 768))`, `Dropout(0.1)`, `Dense(4, activation="softmax")`. Kompilacja: `keras.optimizers.AdamW(learning_rate=LR)`, `sparse_categorical_crossentropy`, accuracy.
2. Wyświetlcie `summary()`. Ile parametrów ma model, a ile sama głowa klasyfikacyjna?
3. Wytrenujcie model: `fit` na `X_train` z walidacją na `X_valid`, `epochs=EPOKI`, `batch_size=BATCH`, `verbose=2`, `EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)`. Zmierzcie czas treningu i narysujcie historię.
4. Oceńcie model na walidacji i na teście (`ocen`). Dla testu wypiszcie `classification_report` i narysujcie macierz pomyłek.

Dlaczego learning rate jest tu około 50 razy mniejszy niż domyślny w Adam (0,001)? Dlaczego wystarczą 2 do 3 epoki? Jak wypadła klasa amb w porównaniu z laboratorium 5?

In [ ]:
class EnkoderHerBERT(torch.nn.Module):
    """HerBERT z transformers; zwraca wektor tokenu <s> z ostatniej warstwy (batch, 768)."""

    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("allegro/herbert-base-cased", add_pooling_layer=False)

    def forward(self, wejscia):
        input_ids, attention_mask = wejscia  # TorchModuleWrapper podaje listę jako jeden argument
        wyjscie = self.bert(input_ids=input_ids.long(), attention_mask=attention_mask.long())
        return wyjscie.last_hidden_state[:, 0]

In [ ]:
# TODO: zbuduj_herbert(): dwa keras.Input(shape=(MAX_LEN,), dtype="int32", name=...) (input_ids, attention_mask)
#       h = layers.TorchModuleWrapper(EnkoderHerBERT(), output_shape=(None, 768), name="herbert")([input_ids, attention_mask])
#       Dropout(0.1), Dense(len(KLASY), activation="softmax", name="klasyfikator")
#       compile: keras.optimizers.AdamW(learning_rate=LR), "sparse_categorical_crossentropy", metrics=["accuracy"]
# TODO: keras.utils.set_random_seed(RANDOM_STATE); herbert = zbuduj_herbert(); herbert.summary()
herbert = None

In [ ]:
# TODO: EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)
# TODO: historia_herbert = herbert.fit(X_train, y_train_ft, validation_data=(X_valid, y_valid_ft),
#       epochs=EPOKI, batch_size=BATCH, callbacks=[...], verbose=2); zmierzcie czas (czas_treningu_herbert)
# TODO: rysuj_historie(historia_herbert, "Fine-tuning HerBERT")

In [ ]:
# TODO: ocen("HerBERT", "valid", y_valid_ft, przewidz(herbert, X_valid))
# TODO: proby_test = herbert.predict(X_test, batch_size=64, verbose=0) z pomiarem czasu (czas_testu)
# TODO: pred_test_herbert = proby_test.argmax(axis=1); ocen("HerBERT", "test", y_test, pred_test_herbert)
# TODO: classification_report (target_names=KLASY, digits=3, zero_division=0), macierz pomyłek (rysuj_macierz)

Odpowiedzi do zadania 3:

1. 
2. 
3. 
4. 

## Zadanie 4: Porównanie z laboratorium 5 i analiza błędów

1. Wpiszcie do słownika `WYNIKI_LAB5` swoje wyniki z laboratorium 5 na teście PolEmo 2.0-IN (accuracy, macro F1, liczba parametrów, czas treningu). Zestawcie je w jednej tabeli z HerBERT.
2. Zmierzcie czas predykcji jednej recenzji przez HerBERT (batch złożony z jednej recenzji, kilka powtórzeń, mediana).
3. Wypiszcie kilka recenzji amb, które HerBERT zaklasyfikował błędnie, z prawdopodobieństwami klas. Z którymi klasami amb jest mylona?

Czy kilkadziesiąt razy więcej parametrów przełożyło się na proporcjonalnie lepszy wynik? Kiedy w praktyce wybralibyście TF-IDF z regresją logistyczną, a kiedy HerBERT? Czy błędy na amb to błędy modelu, czy niejednoznaczne etykiety?

In [ ]:
# Wpiszcie własne wyniki z laboratorium 5 (test PolEmo 2.0-IN, 722 recenzje). None oznacza brak wyniku.
# "parametry": liczba parametrów sieci; dla TF-IDF + LR liczba wag regresji (4 klasy x liczba cech + 4).
# "czas treningu [s]": łączny czas fit z laboratorium 5 (zanotujcie, na jakim sprzęcie).
WYNIKI_LAB5 = {
    "TF-IDF + LR":     {"accuracy": None, "macro F1": None, "parametry": None, "czas treningu [s]": None},
    "Embedding + GAP": {"accuracy": None, "macro F1": None, "parametry": None, "czas treningu [s]": None},
    "BiLSTM":          {"accuracy": None, "macro F1": None, "parametry": None, "czas treningu [s]": None},
    "Conv1D":          {"accuracy": None, "macro F1": None, "parametry": None, "czas treningu [s]": None},
}

In [ ]:
# TODO: jedna = [X_test[0][:1], X_test[1][:1]]; rozgrzewka herbert.predict(jedna), potem 5 pomiarów, mediana w ms
# TODO: porownanie = pd.DataFrame(WYNIKI_LAB5).T; dopiszcie wiersz "HerBERT" (accuracy i macro F1 z wyniki,
#       herbert.count_params(), czas_treningu_herbert)

In [ ]:
# TODO: bledy = test_df.assign(pred=pred_test_herbert) plus kolumny p(klasa) z proby_test
# TODO: amb_bledne: prawdziwa klasa amb, przewidziana inna; ile ich jest i jakie klasy przewidziano
# TODO: wypiszcie 4 takie recenzje (pierwsze 400 znaków) z prawdopodobieństwami klas

Odpowiedzi do zadania 4:

1. 
2. 
3. 

## Pytania kontrolne

Pytania są w instrukcji. Odpowiedzi (po jednym do trzech zdań):

1. 
2. 
3. 
4. 
5. 
6. 

# Zadania rozszerzone

## Zadanie A: Mini-transformer od zera

Zbudujcie mały enkoder transformera w samym Keras i wytrenujcie go od zera na PolEmo, na tych samych tokenach HerBERT, ale z losowo zainicjalizowanymi embeddingami. Trenujemy na całym zbiorze treningowym (5783 recenzje), także w trybie skróconym, bo model jest mały.

1. Gotowa warstwa `TokenyIPozycje` dodaje embedding tokenu i embedding pozycji (pozycje 0 do `MAX_LEN - 1`).
2. Napiszcie `blok_enkodera(x, maska)`: `MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.1)` z `attention_mask=maska[:, None, :]`, dropout, połączenie rezydualne i `LayerNormalization`, potem feed-forward (`Dense(128, relu)`, `Dense(64)`), dropout, znowu połączenie rezydualne i `LayerNormalization`.
3. Napiszcie `zbuduj_mini()`: wejścia jak w HerBERT, `TokenyIPozycje(tokenizer.vocab_size, MAX_LEN, 64)`, maska `ops.cast(attention_mask, "bool")`, jeden blok enkodera, `GlobalAveragePooling1D` z maską, `Dropout(0.1)`, `Dense(4, softmax)`. Optymalizator `adam`.
4. Wytrenujcie (`EPOKI_MINI`, `BATCH_MINI`, `EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)`) na pełnym train z walidacją na pełnym valid, oceńcie na teście. Porównajcie liczbę parametrów, czas i wynik z HerBERT i z modelami z laboratorium 5.

HerBERT używa tokenu o indeksie 1 jako paddingu, więc `mask_zero=True` w warstwie Embedding nie zadziała; maskę bierzemy z `attention_mask`. Dlaczego transformer bez pretreningu nie wygrywa z modelami z laboratorium 5? Gdzie jest większość jego parametrów?

In [ ]:
class TokenyIPozycje(layers.Layer):
    """Embedding tokenu plus embedding pozycji (uczony), jak w BERT."""

    def __init__(self, rozmiar_slownika, max_len, wymiar, **kwargs):
        super().__init__(**kwargs)
        self.tokeny = layers.Embedding(rozmiar_slownika, wymiar)
        self.pozycje = layers.Embedding(max_len, wymiar)

    def call(self, ids):
        pozycje = ops.arange(ops.shape(ids)[-1])
        return self.tokeny(ids) + self.pozycje(pozycje)


# pełny zbiór treningowy i walidacyjny (w trybie pełnym to te same tablice co X_train i X_valid)
if X_train is not None:
    X_train_pelny = X_train if len(train_ft) == len(train_df) else koduj(train_df["sentence"])
    X_valid_pelny = X_valid if len(valid_ft) == len(valid_df) else koduj(valid_df["sentence"])
y_train, y_valid = train_df["y"].to_numpy(), valid_df["y"].to_numpy()

In [ ]:
# TODO: blok_enkodera(x, maska, glowy=4, wymiar=64, wymiar_ff=128, dropout=0.1):
#       att = MultiHeadAttention(num_heads=glowy, key_dim=wymiar // glowy, dropout=dropout)(x, x, attention_mask=maska[:, None, :])
#       x = LayerNormalization()(x + Dropout(att)); ff = Dense(wymiar_ff, relu) -> Dense(wymiar)
#       return LayerNormalization()(x + Dropout(ff))
# TODO: zbuduj_mini(wymiar=64): wejścia input_ids i attention_mask, maska = ops.cast(attention_mask, "bool"),
#       TokenyIPozycje(tokenizer.vocab_size, MAX_LEN, wymiar), blok_enkodera, GlobalAveragePooling1D()(x, mask=maska),
#       Dropout(0.1), Dense(len(KLASY), softmax); compile z "adam"
# TODO: keras.utils.set_random_seed(RANDOM_STATE); mini = zbuduj_mini(); mini.summary()
mini = None

In [ ]:
# TODO: EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
# TODO: mini.fit(X_train_pelny, y_train, validation_data=(X_valid_pelny, y_valid), epochs=EPOKI_MINI,
#       batch_size=BATCH_MINI, ...); czas, rysuj_historie
# TODO: ocen("Mini-transformer", "valid", ...) i ocen("Mini-transformer", "test", ...), classification_report

Odpowiedzi do zadania A:

1. 
2. 
3. 

## Zadanie B: HerBERT jako ekstraktor cech

Zamiast dostrajać 124 mln parametrów, można zamrozić HerBERT, raz policzyć wektory recenzji i wytrenować na nich zwykłą regresję logistyczną. Koszt to tylko inferencja, bez propagacji wstecznej.

1. Wczytajcie świeży, niedostrojony HerBERT (gotowa komórka). Funkcja `wektory_herbert` zwraca dla każdej recenzji wektor tokenu `<s>` i średnią wektorów wszystkich tokenów; uzupełnijcie w niej uśrednianie z pominięciem paddingu.
2. Policzcie wektory dla recenzji treningowych (`N_CECHY`, w trybie pełnym wszystkie), `valid_ft` i testu. Zmierzcie czas.
3. Dla obu wariantów (CLS i średnia) wytrenujcie `make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))` dla C z (0,01, 0,1, 1), wybierzcie wariant i C na walidacji i dopiero wtedy policzcie wynik na teście.
4. Dla chętnych: test PolEmo 2.0-OUT (gotowa komórka wczytuje 494 recenzje z innych dziedzin). Oceńcie na nim dostrojony HerBERT i regresję na cechach, porównajcie z wynikami TF-IDF i Embedding + GAP z zadania B laboratorium 5, jeśli je macie.

Czemu wektor `<s>` niedostrojonego modelu może być gorszą reprezentacją recenzji niż średnia po tokenach? Ile tracimy względem pełnego fine-tuningu i ile zyskujemy na czasie?

In [ ]:
urzadzenie = "cuda" if torch.cuda.is_available() else "cpu"
herbert_zamrozony = AutoModel.from_pretrained("allegro/herbert-base-cased", add_pooling_layer=False).to(urzadzenie)
herbert_zamrozony.eval()


def wektory_herbert(teksty, batch=32):
    """Zwraca (wektory <s>, średnie wektory tokenów bez paddingu), każda macierz (liczba tekstów, 768)."""
    teksty = list(teksty)
    kolejnosc = np.argsort([len(t) for t in teksty])  # podobne długości w batchu, mniej paddingu
    cls = np.zeros((len(teksty), 768), dtype="float32")
    srednie = np.zeros_like(cls)
    with torch.no_grad():
        for i in range(0, len(teksty), batch):
            idx = kolejnosc[i:i + batch]
            e = tokenizer([teksty[j] for j in idx], padding=True, truncation=True, max_length=MAX_LEN,
                          return_tensors="pt").to(urzadzenie)
            h = herbert_zamrozony(**e).last_hidden_state  # (batch, długość, 768)
            m = e["attention_mask"].unsqueeze(-1).to(h.dtype)  # 1 dla tokenów, 0 dla paddingu
            cls[idx] = h[:, 0].cpu().numpy()
            srednie[idx] = 0  # TODO: średnia z h po osi długości, tylko tam, gdzie m == 1 ((h * m).sum ... / m.sum ...)
    return cls, srednie

In [ ]:
# TODO: train_cechy = podzbior(train_df, N_CECHY)
# TODO: cechy = {"train": wektory_herbert(train_cechy["sentence"]), "valid": ..., "test": ...} z pomiarem czasu

In [ ]:
# TODO: dla wariantu i (0 = CLS, 1 = średnia) i C w (0.01, 0.1, 1.0):
#       make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(cechy["train"][i], train_cechy["y"])
#       accuracy i macro F1 na cechy["valid"][i]
# TODO: wybór najlepszego na walidacji, dopiero potem ocen(..., "test", ...) i classification_report

In [ ]:
URL_OUT = "https://huggingface.co/datasets/allegro/klej-polemo2-out/resolve/main/test.csv"
out_df = pd.read_csv(URL_OUT)
out_df["klasa"] = out_df["target"].map(ETYKIETY)
out_df["y"] = out_df["klasa"].map(KLASY.index)
y_out = out_df["y"].to_numpy()
print("test OUT:", len(out_df), "recenzji |", out_df["klasa"].value_counts().reindex(KLASY, fill_value=0).to_dict())

In [ ]:
# TODO (dla chętnych): X_out = koduj(out_df["sentence"]); przewidywania dostrojonego HerBERT i regresji na cechach
# TODO: ocen(nazwa, "test OUT", y_out, pred) i macro F1 dla labels=[0, 1, 2] (klasa zero ma w OUT jedną recenzję)

In [ ]:
# TODO: tabela_wynikow("test")

Odpowiedzi do zadania B:

1. 
2. 
3. 

In [ ]:
print(f"Czas wykonania notebooka od komórki importów: {(time.time() - T_START) / 60:.1f} min")